# Shopify App Marketplace Analysis — Data Cleaning and Transformation
# Shopify 应用市场分析——数据清洗与转换

## Phase 3
## 阶段 3

This notebook cleans and transforms the seven raw datasets based on the issues documented during the data audit. Raw files will not be overwritten.

本 Notebook 根据数据审计阶段记录的问题，对七张原始数据表进行清洗和转换。原始文件不会被覆盖。

In [1]:
import pandas as pd

In [2]:
# Raw data folder / 原始数据文件夹
raw_path = r"D:\xiongsongsong\Programming Language\Data Analyst Projects\shopify-app-marketplace-analysis\data\raw"

# Load datasets / 读取数据
apps = pd.read_csv(
    raw_path + r"\apps.csv",
    encoding="cp1250"
)

apps_categories = pd.read_csv(
    raw_path + r"\apps_categories.csv"
)

categories = pd.read_csv(
    raw_path + r"\categories.csv"
)

key_benefits = pd.read_csv(
    raw_path + r"\key_benefits.csv"
)

pricing_plans = pd.read_csv(
    raw_path + r"\pricing_plans.csv"
)

pricing_plan_features = pd.read_csv(
    raw_path + r"\pricing_plan_features.csv",
    encoding="cp1252"
)

reviews = pd.read_csv(
    raw_path + r"\reviews.csv"
)

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [3]:
print("apps:", apps.shape)
print("apps_categories:", apps_categories.shape)
print("categories:", categories.shape)
print("key_benefits:", key_benefits.shape)
print("pricing_plans:", pricing_plans.shape)
print("pricing_plan_features:", pricing_plan_features.shape)
print("reviews:", reviews.shape)

apps: (11951, 13)
apps_categories: (110830, 2)
categories: (1890, 2)
key_benefits: (47367, 3)
pricing_plans: (20872, 4)
pricing_plan_features: (75778, 3)
reviews: (1133555, 8)


## 1. Clean `apps`
## 1. 清洗 `apps`

The `apps` table is cleaned by removing the completely empty `tagline` field, converting `lastmod` to datetime, and creating a flag that identifies whether an app has reviews.

`apps` 表的清洗内容包括：删除完全为空的 `tagline` 字段、将 `lastmod` 转换为日期，并新增用于判断 App 是否拥有评论的标记字段。

In [4]:
# Create a cleaning copy / 创建清洗副本
apps_clean = apps.copy()

print("Before cleaning:", apps_clean.shape)

Before cleaning: (11951, 13)


In [5]:
# Drop the completely empty column
# 删除完全为空的字段
apps_clean = apps_clean.drop(columns=["tagline"])

# Convert lastmod from string to datetime
# 将 lastmod 从字符串转换为日期
apps_clean["lastmod"] = pd.to_datetime(
    apps_clean["lastmod"],
    errors="coerce"
)

# Create a review-availability flag
# 创建是否拥有评论的标记
apps_clean["has_reviews"] = apps_clean["reviews_count"] > 0

print("Cleaning completed.")

Cleaning completed.


In [6]:
# Validation / 清洗结果验证

print("After cleaning:", apps_clean.shape)
print("Duplicate IDs:", apps_clean["id"].duplicated().sum())
print("Missing lastmod:", apps_clean["lastmod"].isna().sum())
print("Missing values:")
display(apps_clean.isna().sum())

print("\nReview flag distribution:")
display(apps_clean["has_reviews"].value_counts())

After cleaning: (11951, 13)
Duplicate IDs: 0
Missing lastmod: 0
Missing values:


id                 0
url                0
title              0
developer          0
developer_link     0
icon               0
rating             0
reviews_count      0
description_raw    0
description        0
pricing_hint       0
lastmod            0
has_reviews        0
dtype: int64


Review flag distribution:


has_reviews
True     7937
False    4014
Name: count, dtype: int64

In [7]:
# Check whether zero ratings correspond to apps without reviews
# 检查评分为0的App是否都没有评论

rating_review_mismatch = apps_clean[
    ((apps_clean["rating"] == 0) & (apps_clean["has_reviews"])) |
    ((apps_clean["rating"] > 0) & (~apps_clean["has_reviews"]))
]

print("Rating-review mismatches:", len(rating_review_mismatch))

Rating-review mismatches: 0


In [8]:
# Processed data folder / 处理后数据文件夹
processed_path = r"D:\xiongsongsong\Programming Language\Data Analyst Projects\shopify-app-marketplace-analysis\data\processed"

# Export cleaned dataset / 导出清洗后的数据
apps_clean.to_csv(
    processed_path + r"\apps_clean.csv",
    index=False,
    encoding="utf-8"
)

print("apps_clean.csv exported successfully.")

apps_clean.csv exported successfully.


## 2. Clean `apps_categories`
## 2. 清洗 `apps_categories`

The `apps_categories` table is a bridge table connecting apps and categories. Since no missing values were identified during the data audit, cleaning focuses on checking duplicate app-category relationships and validating referential integrity.

`apps_categories` 表是连接 App 和 Category 的桥接表。由于数据审计阶段未发现缺失值，本阶段主要检查重复的 App-Category 关系，并验证外键关联的完整性。

In [9]:
# Create a cleaning copy / 创建清洗副本
apps_categories_clean = apps_categories.copy()

print("Before cleaning:", apps_categories_clean.shape)

Before cleaning: (110830, 2)


In [10]:
# Check duplicate relationships / 检查重复关系
duplicate_relationships = apps_categories_clean.duplicated().sum()

print("Duplicate app-category relationships:", duplicate_relationships)

Duplicate app-category relationships: 3370


In [11]:
# Inspect duplicate relationships / 查看重复关系
duplicate_rows = apps_categories_clean[
    apps_categories_clean.duplicated(keep=False)
]

display(
    duplicate_rows.sort_values(
        by=["app_id", "category_id"]
    ).head(20)
)

,app_id,category_id
11760,003afe73-57ee-401d-bf43-abd32f990965,56ca5de714b3220291ccc556451bdd3c
11777,003afe73-57ee-401d-bf43-abd32f990965,56ca5de714b3220291ccc556451bdd3c
11756,003afe73-57ee-401d-bf43-abd32f990965,f426dc92a8b61dc26f1e6aee37f86ee0
11771,003afe73-57ee-401d-bf43-abd32f990965,f426dc92a8b61dc26f1e6aee37f86ee0
20,007c0e10-12ed-4dde-a7a7-79f4b2b155f4,a96642859a4c6e824fdd0be48fc028be
25,007c0e10-12ed-4dde-a7a7-79f4b2b155f4,a96642859a4c6e824fdd0be48fc028be
66450,00825252-03ad-4567-958d-18b9e9f3284f,f426dc92a8b61dc26f1e6aee37f86ee0
66458,00825252-03ad-4567-958d-18b9e9f3284f,f426dc92a8b61dc26f1e6aee37f86ee0
5916,00b8a5c7-f436-4768-b662-5103e490f55a,2c92b7d273fcc86926be1afb1c40335f
5933,00b8a5c7-f436-4768-b662-5103e490f55a,2c92b7d273fcc86926be1afb1c40335f


In [12]:
# Remove duplicate app-category relationships / 删除重复的 App-Category 关系
apps_categories_clean = apps_categories_clean.drop_duplicates()

print("After removing duplicates:", apps_categories_clean.shape)

After removing duplicates: (107460, 2)


In [13]:
# Validation / 清洗结果验证

print("Shape:", apps_categories_clean.shape)

print("\nMissing values:")
print(apps_categories_clean.isnull().sum())

print("\nDuplicate relationships:")
print(apps_categories_clean.duplicated().sum())

print("\nUnique apps:", apps_categories_clean["app_id"].nunique())
print("Unique categories:", apps_categories_clean["category_id"].nunique())

Shape: (107460, 2)

Missing values:
app_id         0
category_id    0
dtype: int64

Duplicate relationships:
0

Unique apps: 11950
Unique categories: 1889


In [14]:
# Processed data folder / 处理后数据文件夹
processed_path = r"D:\xiongsongsong\Programming Language\Data Analyst Projects\shopify-app-marketplace-analysis\data\processed"

# Export cleaned dataset / 导出清洗后的数据
apps_categories_clean.to_csv(
    processed_path + r"\apps_categories_clean.csv",
    index=False,
    encoding="utf-8"
)

print("apps_categories_clean.csv exported successfully.")

apps_categories_clean.csv exported successfully.


## 3. Clean `categories`
## 3. 清洗 `categories`

The `categories` table stores category-level information for the Shopify App Marketplace. The data audit identified no missing values, duplicate category IDs, or other major data-quality issues. Cleaning therefore focuses on validating the table and preserving its original structure.

`categories` 表存储 Shopify App Marketplace 的分类信息。数据审计阶段未发现缺失值、重复的 Category ID 或其他明显的数据质量问题。因此，本阶段主要对数据进行验证，并保留原始表结构。

In [15]:
# Create a cleaning copy / 创建清洗副本
categories_clean = categories.copy()

print("Before cleaning:", categories_clean.shape)

Before cleaning: (1890, 2)


In [16]:
# Check data quality / 检查数据质量

print("Missing values:")
print(categories_clean.isnull().sum())

print("\nDuplicate rows:")
print(categories_clean.duplicated().sum())

print("\nDuplicate IDs:")
print(categories_clean["id"].duplicated().sum())

Missing values:
id       0
title    0
dtype: int64

Duplicate rows:
0

Duplicate IDs:
1


In [17]:
# Inspect duplicate category IDs / 查看重复的 Category ID
duplicate_category_ids = categories_clean[
    categories_clean["id"].duplicated(keep=False)
]

display(duplicate_category_ids)

,id,title
239,f2d792092fa38504913a64696fb8857e,ALT text
394,f2d792092fa38504913a64696fb8857e,Alt text


In [18]:
# Remove duplicate category ID / 删除重复的 Category ID
categories_clean = categories_clean.drop_duplicates(
    subset=["id"],
    keep="first"
)

print("After cleaning:", categories_clean.shape)

After cleaning: (1889, 2)


In [19]:
# Validation / 清洗结果验证

print("Shape:", categories_clean.shape)

print("\nMissing values:")
print(categories_clean.isnull().sum())

print("\nDuplicate rows:")
print(categories_clean.duplicated().sum())

print("\nDuplicate IDs:")
print(categories_clean["id"].duplicated().sum())

Shape: (1889, 2)

Missing values:
id       0
title    0
dtype: int64

Duplicate rows:
0

Duplicate IDs:
0


In [20]:
# Processed data folder / 处理后数据文件夹
processed_path = r"D:\xiongsongsong\Programming Language\Data Analyst Projects\shopify-app-marketplace-analysis\data\processed"

# Export cleaned dataset / 导出清洗后的数据
categories_clean.to_csv(
    processed_path + r"\categories_clean.csv",
    index=False,
    encoding="utf-8"
)

print("categories_clean.csv exported successfully.")

categories_clean.csv exported successfully.


## 4. Clean `key_benefits`
## 4. 清洗 `key_benefits`

The `key_benefits` table contains the key selling points or benefits associated with each app. The data audit identified that the `title` column is completely missing and that the table contains fully duplicated records. Cleaning therefore focuses on removing the unusable column and eliminating duplicate records.

`key_benefits` 表记录每个 App 的主要卖点或优势。数据审计阶段发现 `title` 列完全缺失，同时表中存在完全重复的记录。因此，本阶段主要删除无法使用的字段，并处理完全重复的数据。

In [21]:
# Create a cleaning copy / 创建清洗副本
key_benefits_clean = key_benefits.copy()

print("Before cleaning:", key_benefits_clean.shape)

Before cleaning: (47367, 3)


In [22]:
# Check known data-quality issues / 检查已知的数据质量问题

print("Missing values:")
print(key_benefits_clean.isnull().sum())

print("\nDuplicate rows:")
print(key_benefits_clean.duplicated().sum())

Missing values:
app_id             0
title          47367
description        0
dtype: int64

Duplicate rows:
15


In [23]:
# Remove unusable column / 删除无法使用的字段
key_benefits_clean = key_benefits_clean.drop(columns=["title"])

print("Columns after removing title:")
print(key_benefits_clean.columns.tolist())

Columns after removing title:
['app_id', 'description']


In [24]:
# Remove duplicate rows / 删除完全重复记录
key_benefits_clean = key_benefits_clean.drop_duplicates()

print("After cleaning:", key_benefits_clean.shape)

After cleaning: (47352, 2)


In [25]:
# Validation / 清洗结果验证

print("Shape:", key_benefits_clean.shape)

print("\nColumns:")
print(key_benefits_clean.columns.tolist())

print("\nMissing values:")
print(key_benefits_clean.isnull().sum())

print("\nDuplicate rows:")
print(key_benefits_clean.duplicated().sum())

Shape: (47352, 2)

Columns:
['app_id', 'description']

Missing values:
app_id         0
description    0
dtype: int64

Duplicate rows:
0


In [26]:
# Processed data folder / 处理后数据文件夹
processed_path = r"D:\xiongsongsong\Programming Language\Data Analyst Projects\shopify-app-marketplace-analysis\data\processed"

# Export cleaned dataset / 导出清洗后的数据
key_benefits_clean.to_csv(
    processed_path + r"\key_benefits_clean.csv",
    index=False,
    encoding="utf-8"
)

print("key_benefits_clean.csv exported successfully.")

key_benefits_clean.csv exported successfully.


## 5. Clean `pricing_plans`
## 5. 清洗 `pricing_plans`

The `pricing_plans` table contains pricing-plan information for Shopify apps. The data audit identified no missing values, duplicate rows, or duplicate plan IDs. However, the `price` field is stored as text and contains different pricing formats such as free plans, free-to-install plans, and monthly subscription prices. Cleaning therefore focuses on validating the table and preparing the pricing field for later analysis.

`pricing_plans` 表记录 Shopify App 的套餐价格信息。数据审计阶段未发现缺失值、完全重复记录或重复的套餐 ID。但是，`price` 字段以文本形式存储，并包含免费套餐、免费安装以及按月收费等不同价格格式。因此，本阶段主要验证数据质量，并为后续价格分析准备 `price` 字段。

In [27]:
# Create a cleaning copy / 创建清洗副本
pricing_plans_clean = pricing_plans.copy()

print("Before cleaning:", pricing_plans_clean.shape)

Before cleaning: (20872, 4)


In [28]:
# Check data quality / 检查数据质量

print("Missing values:")
print(pricing_plans_clean.isnull().sum())

print("\nDuplicate rows:")
print(pricing_plans_clean.duplicated().sum())

print("\nDuplicate IDs:")
print(pricing_plans_clean["id"].duplicated().sum())

print("\nPrice data type:")
print(pricing_plans_clean["price"].dtype)

print("\nSample price values:")
display(pricing_plans_clean["price"].value_counts().head(20))

Missing values:
id        0
app_id    0
title     0
price     0
dtype: int64

Duplicate rows:
0

Duplicate IDs:
0

Price data type:
object

Sample price values:


price
Free               3293
Free to install    2216
$9.99/month         885
$4.99/month         613
$99/month           504
$19.99/month        466
$49/month           451
$29/month           439
$19/month           434
$5/month            386
$10/month           357
$29.99/month        340
$9/month            333
$14.99/month        296
$5.99/month         285
$15/month           267
$2.99/month         248
$20/month           242
$3.99/month         235
$39/month           225
Name: count, dtype: int64

In [29]:
# Inspect pricing formats / 检查价格格式

print("Total unique price values:", pricing_plans_clean["price"].nunique())

print("\nMonthly prices:")
print(pricing_plans_clean["price"].str.contains("/month", case=False, na=False).sum())

print("\nYearly prices:")
print(pricing_plans_clean["price"].str.contains("/year", case=False, na=False).sum())

print("\nFree:")
print((pricing_plans_clean["price"].str.lower() == "free").sum())

print("\nFree to install:")
print((pricing_plans_clean["price"].str.lower() == "free to install").sum())

print("\nOther price formats:")
other_prices = pricing_plans_clean[
    ~pricing_plans_clean["price"].str.contains(
        r"/month|/year|^free$|^free to install$",
        case=False,
        regex=True,
        na=False
    )
]

display(other_prices["price"].value_counts().head(30))

Total unique price values: 932

Monthly prices:
15033

Yearly prices:
252

Free:
3293

Free to install:
2216

Other price formats:


price
$49 one-time charge        6
$5 one-time charge         5
$99 one-time charge        4
$1 one-time charge         4
$10 one-time charge        4
$19 one-time charge        3
$29.99 one-time charge     3
$9.99 one-time charge      3
$39.99 one-time charge     3
$20 one-time charge        3
$99.99 one-time charge     3
$100 one-time charge       2
$25 one-time charge        2
$4.99 one-time charge      2
$3.99 one-time charge      2
$125 one-time charge       1
$69 one-time charge        1
$15 one-time charge        1
$2,000 one-time charge     1
$10,000 one-time charge    1
$59.88 one-time charge     1
$2 one-time charge         1
$169 one-time charge       1
$500 one-time charge       1
$749 one-time charge       1
$30 one-time charge        1
$19.95 one-time charge     1
$199 one-time charge       1
$349 one-time charge       1
$3 one-time charge         1
Name: count, dtype: int64

In [30]:
# Classify price formats / 对价格格式进行分类

def classify_price_format(price):
    price = str(price).lower()

    if price == "free":
        return "free"
    elif price == "free to install":
        return "free_to_install"
    elif "/month" in price:
        return "monthly"
    elif "/year" in price:
        return "yearly"
    elif "one-time charge" in price:
        return "one_time"
    else:
        return "other"

pricing_plans_clean["price_type"] = pricing_plans_clean["price"].apply(
    classify_price_format
)

print("Price type distribution:")
display(pricing_plans_clean["price_type"].value_counts())

Price type distribution:


price_type
monthly            15033
free                3293
free_to_install     2216
yearly               252
one_time              78
Name: count, dtype: int64

In [31]:
# Inspect unclassified price formats / 检查尚未分类的价格格式

other_price_formats = pricing_plans_clean[
    pricing_plans_clean["price_type"] == "other"
]

print("Other price records:", len(other_price_formats))
print("Other unique price values:", other_price_formats["price"].nunique())

display(
    other_price_formats["price"]
    .value_counts()
    .head(30)
)

Other price records: 0
Other unique price values: 0


Series([], Name: count, dtype: int64)

In [32]:
# Extract numeric price amount / 提取数值价格

pricing_plans_clean["price_amount"] = (
    pricing_plans_clean["price"]
    .str.replace(",", "", regex=False)
    .str.extract(r"\$([0-9]+(?:\.[0-9]+)?)")[0]
)

# Convert to numeric / 转换为数值类型
pricing_plans_clean["price_amount"] = pd.to_numeric(
    pricing_plans_clean["price_amount"],
    errors="coerce"
)

# Free plans have a price amount of 0
# 免费套餐的价格金额设为 0
pricing_plans_clean.loc[
    pricing_plans_clean["price_type"].isin(["free", "free_to_install"]),
    "price_amount"
] = 0

In [33]:
# Validate extracted prices / 验证提取后的价格

print("Missing price amounts:")
print(pricing_plans_clean["price_amount"].isna().sum())

print("\nSample results:")
display(
    pricing_plans_clean[
        ["price", "price_type", "price_amount"]
    ].drop_duplicates().head(20)
)

print("\nPrice amount summary:")
display(pricing_plans_clean["price_amount"].describe())

Missing price amounts:
0

Sample results:


,price,price_type,price_amount
0,Free to install,free_to_install,0.00
1,$5.90/month,monthly,5.90
2,Free,free,0.00
3,$4.99/month,monthly,4.99
4,$9.99/month,monthly,9.99
5,$29/month,monthly,29.00
6,$49/month,monthly,49.00
7,$79/month,monthly,79.00
8,$149/month,monthly,149.00
13,$50/month,monthly,50.00



Price amount summary:


count    20872.000000
mean        94.439705
std       1128.816325
min          0.000000
25%          0.000000
50%         10.000000
75%         40.000000
max      99999.000000
Name: price_amount, dtype: float64

In [34]:
# Validation / 清洗结果验证

print("Shape:", pricing_plans_clean.shape)

print("\nMissing values:")
print(pricing_plans_clean.isnull().sum())

print("\nDuplicate rows:")
print(pricing_plans_clean.duplicated().sum())

print("\nDuplicate IDs:")
print(pricing_plans_clean["id"].duplicated().sum())

print("\nPrice type distribution:")
print(pricing_plans_clean["price_type"].value_counts())

print("\nMissing price amounts:")
print(pricing_plans_clean["price_amount"].isna().sum())

Shape: (20872, 6)

Missing values:
id              0
app_id          0
title           0
price           0
price_type      0
price_amount    0
dtype: int64

Duplicate rows:
0

Duplicate IDs:
0

Price type distribution:
price_type
monthly            15033
free                3293
free_to_install     2216
yearly               252
one_time              78
Name: count, dtype: int64

Missing price amounts:
0


In [35]:
# Processed data folder / 处理后数据文件夹
processed_path = r"D:\xiongsongsong\Programming Language\Data Analyst Projects\shopify-app-marketplace-analysis\data\processed"

# Export cleaned dataset / 导出清洗后的数据
pricing_plans_clean.to_csv(
    processed_path + r"\pricing_plans_clean.csv",
    index=False,
    encoding="utf-8"
)

print("pricing_plans_clean.csv exported successfully.")

pricing_plans_clean.csv exported successfully.


## 6. Clean `pricing_plan_features`
## 6. 清洗 `pricing_plan_features`

The `pricing_plan_features` table contains the features associated with individual pricing plans. A pricing plan can contain multiple features, so repeated `pricing_plan_id` and `app_id` values are expected. The data audit identified no missing values but found a small number of fully duplicated records. Cleaning therefore focuses on removing exact duplicates while preserving the original one-to-many relationships.

`pricing_plan_features` 表记录不同定价套餐所包含的功能。由于一个 Pricing Plan 可以包含多个 Feature，因此 `pricing_plan_id` 和 `app_id` 出现重复属于正常情况。数据审计阶段未发现缺失值，但发现少量完全重复记录。因此，本阶段主要删除完全重复的数据，同时保留原有的一对多关系。

In [36]:
# Create a cleaning copy / 创建清洗副本
pricing_plan_features_clean = pricing_plan_features.copy()

print("Before cleaning:", pricing_plan_features_clean.shape)

Before cleaning: (75778, 3)


In [37]:
# Check data quality / 检查数据质量

print("Missing values:")
print(pricing_plan_features_clean.isnull().sum())

print("\nDuplicate rows:")
print(pricing_plan_features_clean.duplicated().sum())

Missing values:
pricing_plan_id    0
app_id             0
feature            0
dtype: int64

Duplicate rows:
1319


In [38]:
# Inspect duplicated records / 查看重复记录
duplicate_feature_rows = pricing_plan_features_clean[
    pricing_plan_features_clean.duplicated(keep=False)
]

print("Rows involved in duplicates:", len(duplicate_feature_rows))

display(
    duplicate_feature_rows
    .sort_values(by=["pricing_plan_id", "app_id", "feature"])
    .head(30)
)

Rows involved in duplicates: 1894


,pricing_plan_id,app_id,feature
25232,001a79ac-8a5a-4aff-afa7-c2f44a396f03,2200bf38-0ab3-46c4-a529-c63f86cf3551,#NAME?
25233,001a79ac-8a5a-4aff-afa7-c2f44a396f03,2200bf38-0ab3-46c4-a529-c63f86cf3551,#NAME?
24861,006965a2-f90b-4591-91eb-647eff55024a,ec6e69d0-2b2c-470b-81d4-f93083d4c347,#NAME?
24862,006965a2-f90b-4591-91eb-647eff55024a,ec6e69d0-2b2c-470b-81d4-f93083d4c347,#NAME?
24863,006965a2-f90b-4591-91eb-647eff55024a,ec6e69d0-2b2c-470b-81d4-f93083d4c347,#NAME?
24864,006965a2-f90b-4591-91eb-647eff55024a,ec6e69d0-2b2c-470b-81d4-f93083d4c347,#NAME?
24865,006965a2-f90b-4591-91eb-647eff55024a,ec6e69d0-2b2c-470b-81d4-f93083d4c347,#NAME?
24866,006965a2-f90b-4591-91eb-647eff55024a,ec6e69d0-2b2c-470b-81d4-f93083d4c347,#NAME?
24867,006965a2-f90b-4591-91eb-647eff55024a,ec6e69d0-2b2c-470b-81d4-f93083d4c347,#NAME?
24868,006965a2-f90b-4591-91eb-647eff55024a,ec6e69d0-2b2c-470b-81d4-f93083d4c347,#NAME?


In [39]:
# Check suspicious #NAME? values / 检查异常的 #NAME? 值

name_error_rows = pricing_plan_features_clean[
    pricing_plan_features_clean["feature"] == "#NAME?"
]

print("Total #NAME? rows:", len(name_error_rows))
print(
    "Unique pricing plans affected:",
    name_error_rows["pricing_plan_id"].nunique()
)
print(
    "Unique apps affected:",
    name_error_rows["app_id"].nunique()
)

print("\n#NAME? rows that are duplicates:")
print(name_error_rows.duplicated().sum())

Total #NAME? rows: 1926
Unique pricing plans affected: 700
Unique apps affected: 326

#NAME? rows that are duplicates:
1226


In [40]:
# Remove invalid #NAME? feature records
# 删除无效的 #NAME? 功能记录

pricing_plan_features_clean = pricing_plan_features_clean[
    pricing_plan_features_clean["feature"] != "#NAME?"
].copy()

print("After removing #NAME? rows:", pricing_plan_features_clean.shape)

print("\nRemaining duplicate rows:")
print(pricing_plan_features_clean.duplicated().sum())

After removing #NAME? rows: (73852, 3)

Remaining duplicate rows:
93


In [41]:
# Remove remaining duplicate records / 删除剩余的完全重复记录
pricing_plan_features_clean = (
    pricing_plan_features_clean
    .drop_duplicates()
)

print("After removing duplicates:", pricing_plan_features_clean.shape)

After removing duplicates: (73759, 3)


In [42]:
# Validation / 清洗结果验证

print("Shape:", pricing_plan_features_clean.shape)

print("\nMissing values:")
print(pricing_plan_features_clean.isnull().sum())

print("\nDuplicate rows:")
print(pricing_plan_features_clean.duplicated().sum())

print("\nInvalid #NAME? values:")
print(
    (pricing_plan_features_clean["feature"] == "#NAME?").sum()
)

Shape: (73759, 3)

Missing values:
pricing_plan_id    0
app_id             0
feature            0
dtype: int64

Duplicate rows:
0

Invalid #NAME? values:
0


In [43]:
# Processed data folder / 处理后数据文件夹
processed_path = r"D:\xiongsongsong\Programming Language\Data Analyst Projects\shopify-app-marketplace-analysis\data\processed"

# Export cleaned dataset / 导出清洗后的数据
pricing_plan_features_clean.to_csv(
    processed_path + r"\pricing_plan_features_clean.csv",
    index=False,
    encoding="utf-8"
)

print("pricing_plan_features_clean.csv exported successfully.")

pricing_plan_features_clean.csv exported successfully.


## 7. Clean `reviews`
## 7. 清洗 `reviews`

The `reviews` table contains user-review information for Shopify apps and is the largest dataset in the project. The data audit identified several cleaning issues, including a completely missing `helpful_count` field, missing review text, missing developer replies, and date fields stored as text. Some `posted_at` values also contain the prefix `Edited`.

Cleaning therefore focuses on removing unusable fields, standardizing date variables, preserving meaningful missing values, and creating additional fields for later review and developer-response analysis.

`reviews` 表记录 Shopify App 的用户评论信息，也是本项目中规模最大的数据表。数据审计阶段发现多个需要处理的问题，包括 `helpful_count` 字段完全缺失、部分评论正文缺失、部分开发者回复缺失，以及日期字段以文本形式存储。此外，部分 `posted_at` 值还包含 `Edited` 前缀。

因此，本阶段主要删除无法使用的字段、统一日期格式、保留具有实际含义的缺失值，并创建用于后续评论分析和开发者回复分析的辅助字段。

In [44]:
# Create a cleaning copy / 创建清洗副本
reviews_clean = reviews.copy()

print("Before cleaning:", reviews_clean.shape)

Before cleaning: (1133555, 8)


In [45]:
# Check known data-quality issues / 检查已知的数据质量问题

print("Missing values:")
print(reviews_clean.isnull().sum())

print("\nDuplicate rows:")
print(reviews_clean.duplicated().sum())

print("\nRating range:")
print(reviews_clean["rating"].min(), reviews_clean["rating"].max())

print("\nData types:")
print(reviews_clean.dtypes)

Missing values:
app_id                             0
author                            80
rating                             0
posted_at                          0
body                           44130
helpful_count                1133555
developer_reply               811971
developer_reply_posted_at     811971
dtype: int64

Duplicate rows:
0

Rating range:
1 5

Data types:
app_id                        object
author                        object
rating                         int64
posted_at                     object
body                          object
helpful_count                float64
developer_reply               object
developer_reply_posted_at     object
dtype: object


In [46]:
# Remove completely empty column / 删除完全为空的字段
reviews_clean = reviews_clean.drop(columns=["helpful_count"])

print("Shape after removing helpful_count:", reviews_clean.shape)
print("Columns:")
print(reviews_clean.columns.tolist())

Shape after removing helpful_count: (1133555, 7)
Columns:
['app_id', 'author', 'rating', 'posted_at', 'body', 'developer_reply', 'developer_reply_posted_at']


In [47]:
# Inspect posted_at values / 检查 posted_at 日期格式

print("Sample posted_at values:")
display(reviews_clean["posted_at"].head(20))

edited_mask = reviews_clean["posted_at"].str.contains(
    "Edited",
    case=False,
    na=False
)

print("\nRows containing 'Edited':")
print(edited_mask.sum())

print("\nSample Edited values:")
display(
    reviews_clean.loc[edited_mask, "posted_at"].head(20)
)

Sample posted_at values:


0               June 28, 2021
1     Edited October 11, 2021
2              April 30, 2024
3         Edited May 26, 2020
4       Edited August 6, 2020
5         Edited June 9, 2020
6              March 28, 2023
7               June 21, 2019
8           December 28, 2019
9              March 18, 2022
10             April 29, 2020
11        Edited May 20, 2020
12               May 15, 2024
13           October 16, 2024
14           October 16, 2024
15           October 10, 2024
16             April 10, 2024
17             April 19, 2021
18              July 15, 2022
19             April 13, 2022
Name: posted_at, dtype: object


Rows containing 'Edited':
103656

Sample Edited values:


1        Edited October 11, 2021
3            Edited May 26, 2020
4          Edited August 6, 2020
5            Edited June 9, 2020
11           Edited May 20, 2020
20     Edited September 22, 2021
21          Edited April 3, 2021
33           Edited May 10, 2019
34        Edited August 17, 2019
35       Edited January 10, 2020
44           Edited May 23, 2023
47      Edited September 9, 2021
50           Edited May 17, 2023
68         Edited March 24, 2022
86          Edited June 20, 2019
107           Edited May 9, 2019
109      Edited January 18, 2020
115       Edited August 12, 2020
117     Edited December 13, 2021
138     Edited November 11, 2024
Name: posted_at, dtype: object

In [48]:
# Create edited-review flag / 创建评论是否被编辑的标记
reviews_clean["is_edited"] = reviews_clean["posted_at"].str.contains(
    "Edited",
    case=False,
    na=False
)

# Remove "Edited" prefix / 删除 Edited 前缀
reviews_clean["posted_at"] = (
    reviews_clean["posted_at"]
    .str.replace(r"^Edited\s+", "", regex=True)
    .str.strip()
)

# Convert to datetime / 转换为日期类型
reviews_clean["posted_at"] = pd.to_datetime(
    reviews_clean["posted_at"],
    errors="coerce"
)

In [49]:
# Validate posted_at cleaning / 验证 posted_at 清洗结果

print("posted_at data type:")
print(reviews_clean["posted_at"].dtype)

print("\nMissing posted_at after conversion:")
print(reviews_clean["posted_at"].isna().sum())

print("\nEdited review distribution:")
print(reviews_clean["is_edited"].value_counts())

print("\nDate range:")
print("Earliest:", reviews_clean["posted_at"].min())
print("Latest:", reviews_clean["posted_at"].max())

posted_at data type:
datetime64[ns]

Missing posted_at after conversion:
0

Edited review distribution:
is_edited
False    1029899
True      103656
Name: count, dtype: int64

Date range:
Earliest: 2009-06-02 00:00:00
Latest: 2024-11-25 00:00:00


In [50]:
# Inspect developer reply dates / 检查开发者回复日期格式

reply_date_non_null = reviews_clean[
    reviews_clean["developer_reply_posted_at"].notna()
]

print("Non-null developer reply dates:")
print(len(reply_date_non_null))

print("\nSample developer reply dates:")
display(
    reply_date_non_null["developer_reply_posted_at"].head(20)
)

print("\nRows containing 'Edited':")
print(
    reply_date_non_null["developer_reply_posted_at"]
    .str.contains("Edited", case=False, na=False)
    .sum()
)

Non-null developer reply dates:
321584

Sample developer reply dates:


0       October 11, 2021
1       October 11, 2021
2            May 8, 2024
4        August 14, 2020
6         March 28, 2023
9          April 7, 2022
12          May 16, 2024
37          July 6, 2020
68     February 28, 2022
71     February 20, 2024
73      October 15, 2023
74       January 9, 2024
75       January 9, 2024
77     February 22, 2024
79     February 22, 2024
90         April 7, 2022
96     February 20, 2024
116        March 8, 2021
117    December 13, 2021
118    December 12, 2021
Name: developer_reply_posted_at, dtype: object


Rows containing 'Edited':
0


In [51]:
# Convert developer reply date to datetime
# 将开发者回复日期转换为日期类型

reviews_clean["developer_reply_posted_at"] = pd.to_datetime(
    reviews_clean["developer_reply_posted_at"],
    errors="coerce"
)

In [52]:
# Create developer reply flag / 创建开发者是否回复的标记

reviews_clean["has_developer_reply"] = (
    reviews_clean["developer_reply"].notna()
)

In [53]:
# Check consistency between developer reply and reply date
# 检查开发者回复和回复日期是否一致

reply_date_mismatch = reviews_clean[
    (
        reviews_clean["developer_reply"].notna() &
        reviews_clean["developer_reply_posted_at"].isna()
    )
    |
    (
        reviews_clean["developer_reply"].isna() &
        reviews_clean["developer_reply_posted_at"].notna()
    )
]

print("Reply-date mismatches:", len(reply_date_mismatch))

print("\nDeveloper reply distribution:")
print(reviews_clean["has_developer_reply"].value_counts())

print("\nMissing developer reply dates:")
print(reviews_clean["developer_reply_posted_at"].isna().sum())

Reply-date mismatches: 0

Developer reply distribution:
has_developer_reply
False    811971
True     321584
Name: count, dtype: int64

Missing developer reply dates:
811971


In [54]:
# Create review-text availability flag
# 创建评论正文是否可用的标记

reviews_clean["has_review_text"] = (
    reviews_clean["body"].notna()
)

print("Review text distribution:")
print(reviews_clean["has_review_text"].value_counts())

Review text distribution:
has_review_text
True     1089425
False      44130
Name: count, dtype: int64


In [55]:
# Check missing authors / 检查缺失作者

print("Missing authors:")
print(reviews_clean["author"].isna().sum())

Missing authors:
80


In [56]:
# Inspect reviews without review text
# 检查没有评论正文的记录

display(
    reviews_clean.loc[
        ~reviews_clean["has_review_text"],
        ["app_id", "author", "rating", "posted_at", "body"]
    ].head(20)
)

print("\nRating distribution for reviews without text:")
print(
    reviews_clean.loc[
        ~reviews_clean["has_review_text"],
        "rating"
    ].value_counts().sort_index()
)

,app_id,author,rating,posted_at,body
77,5ebc36fe-934b-4875-bcad-fb11185361e9,Don't Buy Tomorrow,5,2024-02-19,NaN
78,5ebc36fe-934b-4875-bcad-fb11185361e9,Feame,5,2024-03-05,NaN
79,5ebc36fe-934b-4875-bcad-fb11185361e9,Y-select,5,2023-12-11,NaN
81,007c0e10-12ed-4dde-a7a7-79f4b2b155f4,Arconautide pood,5,2024-03-17,NaN
96,5ebc36fe-934b-4875-bcad-fb11185361e9,SHOPMINIPRICE,5,2024-02-19,NaN
545,e31e7f05-f48f-45c1-b5c6-686f370c5a38,www.Shopthatapp.com,5,2024-08-14,NaN
546,e31e7f05-f48f-45c1-b5c6-686f370c5a38,Eco Nurtura,5,2024-07-06,NaN
557,e31e7f05-f48f-45c1-b5c6-686f370c5a38,Diamond Princess Boutique,5,2023-12-13,NaN
558,e31e7f05-f48f-45c1-b5c6-686f370c5a38,Curated Corner,5,2024-03-25,NaN
738,bb0faaf6-71e4-4753-b8b3-a86c576f02fb,Aqouh Jarar,5,2024-09-05,NaN



Rating distribution for reviews without text:
rating
1      365
2      146
3      587
4     3137
5    39895
Name: count, dtype: int64


In [57]:
# Final validation / 最终清洗结果验证

print("Shape:", reviews_clean.shape)

print("\nColumns:")
print(reviews_clean.columns.tolist())

print("\nMissing values:")
print(reviews_clean.isnull().sum())

print("\nDuplicate rows:")
print(reviews_clean.duplicated().sum())

print("\nRating range:")
print(
    reviews_clean["rating"].min(),
    reviews_clean["rating"].max()
)

print("\nData types:")
print(reviews_clean.dtypes)

print("\nEdited review distribution:")
print(reviews_clean["is_edited"].value_counts())

print("\nDeveloper reply distribution:")
print(reviews_clean["has_developer_reply"].value_counts())

print("\nReview text distribution:")
print(reviews_clean["has_review_text"].value_counts())

Shape: (1133555, 10)

Columns:
['app_id', 'author', 'rating', 'posted_at', 'body', 'developer_reply', 'developer_reply_posted_at', 'is_edited', 'has_developer_reply', 'has_review_text']

Missing values:
app_id                            0
author                           80
rating                            0
posted_at                         0
body                          44130
developer_reply              811971
developer_reply_posted_at    811971
is_edited                         0
has_developer_reply               0
has_review_text                   0
dtype: int64

Duplicate rows:
0

Rating range:
1 5

Data types:
app_id                               object
author                               object
rating                                int64
posted_at                    datetime64[ns]
body                                 object
developer_reply                      object
developer_reply_posted_at    datetime64[ns]
is_edited                              bool
has_developer_reply  

In [58]:
# Processed data folder / 处理后数据文件夹
processed_path = r"D:\xiongsongsong\Programming Language\Data Analyst Projects\shopify-app-marketplace-analysis\data\processed"

# Export cleaned dataset / 导出清洗后的数据
reviews_clean.to_csv(
    processed_path + r"\reviews_clean.csv",
    index=False,
    encoding="utf-8"
)

print("reviews_clean.csv exported successfully.")

reviews_clean.csv exported successfully.


# Final Cross-Table Validation
# 最终跨表验证

After cleaning all seven datasets, final validation is performed to ensure that primary keys remain unique and relationships between tables remain valid.

完成七张数据表的清洗后，进行最终跨表验证，以确保主键仍保持唯一，并确认各数据表之间的关联关系没有因清洗过程而被破坏。

In [59]:
# Check primary-key uniqueness / 检查主键唯一性

print("apps duplicate IDs:")
print(apps_clean["id"].duplicated().sum())

print("\ncategories duplicate IDs:")
print(categories_clean["id"].duplicated().sum())

print("\npricing_plans duplicate IDs:")
print(pricing_plans_clean["id"].duplicated().sum())

apps duplicate IDs:
0

categories duplicate IDs:
0

pricing_plans duplicate IDs:
0


In [60]:
# Check app_id referential integrity
# 检查 app_id 外键完整性

valid_app_ids = set(apps_clean["id"])

tables_with_app_id = {
    "apps_categories": apps_categories_clean,
    "key_benefits": key_benefits_clean,
    "pricing_plans": pricing_plans_clean,
    "pricing_plan_features": pricing_plan_features_clean,
    "reviews": reviews_clean
}

for table_name, df in tables_with_app_id.items():
    invalid_count = (~df["app_id"].isin(valid_app_ids)).sum()
    print(f"{table_name} invalid app_id: {invalid_count}")

apps_categories invalid app_id: 0
key_benefits invalid app_id: 0
pricing_plans invalid app_id: 0
pricing_plan_features invalid app_id: 0
reviews invalid app_id: 0


In [61]:
# Check remaining foreign-key relationships
# 检查剩余的外键关系

# apps_categories.category_id -> categories.id
invalid_category_ids = (
    ~apps_categories_clean["category_id"]
    .isin(categories_clean["id"])
).sum()

print("apps_categories invalid category_id:", invalid_category_ids)

# pricing_plan_features.pricing_plan_id -> pricing_plans.id
invalid_pricing_plan_ids = (
    ~pricing_plan_features_clean["pricing_plan_id"]
    .isin(pricing_plans_clean["id"])
).sum()

print(
    "pricing_plan_features invalid pricing_plan_id:",
    invalid_pricing_plan_ids
)

apps_categories invalid category_id: 0
pricing_plan_features invalid pricing_plan_id: 0


In [62]:
# Final dataset summary / 最终数据集汇总

final_summary = pd.DataFrame({
    "table": [
        "apps",
        "apps_categories",
        "categories",
        "key_benefits",
        "pricing_plans",
        "pricing_plan_features",
        "reviews"
    ],
    "rows": [
        len(apps_clean),
        len(apps_categories_clean),
        len(categories_clean),
        len(key_benefits_clean),
        len(pricing_plans_clean),
        len(pricing_plan_features_clean),
        len(reviews_clean)
    ],
    "columns": [
        apps_clean.shape[1],
        apps_categories_clean.shape[1],
        categories_clean.shape[1],
        key_benefits_clean.shape[1],
        pricing_plans_clean.shape[1],
        pricing_plan_features_clean.shape[1],
        reviews_clean.shape[1]
    ]
})

display(final_summary)

,table,rows,columns
0,apps,11951,13
1,apps_categories,107460,2
2,categories,1889,2
3,key_benefits,47352,2
4,pricing_plans,20872,6
5,pricing_plan_features,73759,3
6,reviews,1133555,10


In [63]:
print(apps_clean.columns.tolist())

print("\nData types:")
print(apps_clean.dtypes)

['id', 'url', 'title', 'developer', 'developer_link', 'icon', 'rating', 'reviews_count', 'description_raw', 'description', 'pricing_hint', 'lastmod', 'has_reviews']

Data types:
id                         object
url                        object
title                      object
developer                  object
developer_link             object
icon                       object
rating                    float64
reviews_count               int64
description_raw            object
description                object
pricing_hint               object
lastmod            datetime64[ns]
has_reviews                  bool
dtype: object


In [64]:
print(key_benefits_clean.columns.tolist())
print(key_benefits_clean.dtypes)

['app_id', 'description']
app_id         object
description    object
dtype: object


In [65]:
print(pricing_plans_clean.columns.tolist())
print(pricing_plans_clean.dtypes)

['id', 'app_id', 'title', 'price', 'price_type', 'price_amount']
id               object
app_id           object
title            object
price            object
price_type       object
price_amount    float64
dtype: object


In [66]:
print(pricing_plan_features_clean.columns.tolist())
print(pricing_plan_features_clean.dtypes)

['pricing_plan_id', 'app_id', 'feature']
pricing_plan_id    object
app_id             object
feature            object
dtype: object


In [67]:
print(reviews_clean.columns.tolist())
print(reviews_clean.dtypes)

['app_id', 'author', 'rating', 'posted_at', 'body', 'developer_reply', 'developer_reply_posted_at', 'is_edited', 'has_developer_reply', 'has_review_text']
app_id                               object
author                               object
rating                                int64
posted_at                    datetime64[ns]
body                                 object
developer_reply                      object
developer_reply_posted_at    datetime64[ns]
is_edited                              bool
has_developer_reply                    bool
has_review_text                        bool
dtype: object


In [68]:
print("Rows:", len(reviews_clean))

print(
    "Missing app_id:",
    reviews_clean["app_id"].isna().sum()
)

print(
    "Blank app_id:",
    reviews_clean["app_id"].astype(str).str.strip().eq("").sum()
)

Rows: 1133555
Missing app_id: 0
Blank app_id: 0


In [69]:
reviews_csv_check = pd.read_csv(
    processed_path + r"\reviews_clean.csv"
)

print("CSV rows:", len(reviews_csv_check))
print("Missing app_id:", reviews_csv_check["app_id"].isna().sum())
print(
    "Blank app_id:",
    reviews_csv_check["app_id"].astype(str).str.strip().eq("").sum()
)

CSV rows: 1133555
Missing app_id: 0
Blank app_id: 0


In [70]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [71]:
conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 18 for SQL Server};"
    r"SERVER=.\SQLEXPRESS;"
    r"DATABASE=ShopifyMarketplace;"
    r"Trusted_Connection=yes;"
    r"TrustServerCertificate=yes;"
)

print("Connected successfully!")

Connected successfully!


In [72]:
cursor = conn.cursor()
cursor.fast_executemany = True

insert_sql = """
INSERT INTO reviews (
    app_id,
    author,
    rating,
    posted_at,
    body,
    developer_reply,
    developer_reply_posted_at,
    is_edited,
    has_developer_reply,
    has_review_text
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

test_df = reviews_clean.head(1000).copy()

test_df = test_df.astype(object).where(test_df.notna(), None)

rows = list(test_df.itertuples(index=False, name=None))

cursor.executemany(insert_sql, rows)
conn.commit()

print("Inserted:", len(rows))

Inserted: 1000


In [73]:
batch_size = 10000
total_rows = len(reviews_clean)

for start in range(0, total_rows, batch_size):
    end = min(start + batch_size, total_rows)

    batch = reviews_clean.iloc[start:end].copy()
    batch = batch.astype(object).where(batch.notna(), None)

    rows = list(batch.itertuples(index=False, name=None))

    cursor.executemany(insert_sql, rows)
    conn.commit()

    print(f"Inserted {end:,} / {total_rows:,}")

print("Import complete!")

Inserted 10,000 / 1,133,555
Inserted 20,000 / 1,133,555
Inserted 30,000 / 1,133,555
Inserted 40,000 / 1,133,555
Inserted 50,000 / 1,133,555
Inserted 60,000 / 1,133,555
Inserted 70,000 / 1,133,555
Inserted 80,000 / 1,133,555
Inserted 90,000 / 1,133,555
Inserted 100,000 / 1,133,555
Inserted 110,000 / 1,133,555
Inserted 120,000 / 1,133,555
Inserted 130,000 / 1,133,555
Inserted 140,000 / 1,133,555
Inserted 150,000 / 1,133,555
Inserted 160,000 / 1,133,555
Inserted 170,000 / 1,133,555
Inserted 180,000 / 1,133,555
Inserted 190,000 / 1,133,555
Inserted 200,000 / 1,133,555
Inserted 210,000 / 1,133,555
Inserted 220,000 / 1,133,555
Inserted 230,000 / 1,133,555
Inserted 240,000 / 1,133,555
Inserted 250,000 / 1,133,555
Inserted 260,000 / 1,133,555
Inserted 270,000 / 1,133,555
Inserted 280,000 / 1,133,555
Inserted 290,000 / 1,133,555
Inserted 300,000 / 1,133,555
Inserted 310,000 / 1,133,555
Inserted 320,000 / 1,133,555
Inserted 330,000 / 1,133,555
Inserted 340,000 / 1,133,555
Inserted 350,000 / 1,13